# 面试问题：模型发布怎样串联 Shadow、Canary、分群门禁与自动 Rollback？

        ## 可直接复述的回答主线

        1. Shadow 让候选模型处理真实流量副本但不影响用户，用来检查 schema、错误率、延迟和离线标签表现。
2. Shadow 通过后才按稳定哈希把小比例用户送到 Canary，避免同一用户在模型间来回跳。
3. 推广门禁不能只看全局准确率，还要看错误率、p95 延迟和关键分群回归。
4. 任一硬指标越界就触发 Rollback，切回 champion 并记录触发指标、版本和暴露请求。
5. 同一批请求应展示 champion/candidate 预测、shadow 指标、canary bucket、用户暴露和状态迁移。
6. 生产还需最小样本、置信区间、指标延迟、配置原子切换、模型缓存预热、回滚演练和数据版本审计。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是十二笔欺诈审核请求，包含 regular/vip 分群、标签、champion 与 candidate 分数、两套延迟及候选在线错误。候选全局离线准确率更高，但 vip 分群退化且在线尾延迟变差，用于真实触发灰度回滚。

In [1]:
import hashlib  # 对 request_id 计算跨进程稳定的 Canary bucket。
import math  # 计算离散样本 p95 所需的向上取整索引。
requests = [{"id": "deploy-01", "segment": "regular", "label": 1, "champion_prob": 0.80, "candidate_prob": 0.90, "champion_ms": 72, "candidate_ms": 84, "candidate_error": False}, {"id": "deploy-02", "segment": "regular", "label": 0, "champion_prob": 0.70, "candidate_prob": 0.20, "champion_ms": 70, "candidate_ms": 89, "candidate_error": False}, {"id": "deploy-03", "segment": "regular", "label": 1, "champion_prob": 0.40, "candidate_prob": 0.80, "champion_ms": 75, "candidate_ms": 93, "candidate_error": False}, {"id": "deploy-04", "segment": "regular", "label": 0, "champion_prob": 0.20, "candidate_prob": 0.10, "champion_ms": 68, "candidate_ms": 87, "candidate_error": False}, {"id": "deploy-05", "segment": "regular", "label": 1, "champion_prob": 0.70, "candidate_prob": 0.80, "champion_ms": 80, "candidate_ms": 96, "candidate_error": False}, {"id": "deploy-06", "segment": "regular", "label": 0, "champion_prob": 0.60, "candidate_prob": 0.20, "champion_ms": 74, "candidate_ms": 130, "candidate_error": True}, {"id": "deploy-07", "segment": "regular", "label": 1, "champion_prob": 0.80, "candidate_prob": 0.90, "champion_ms": 76, "candidate_ms": 91, "candidate_error": False}, {"id": "deploy-08", "segment": "regular", "label": 0, "champion_prob": 0.20, "candidate_prob": 0.10, "champion_ms": 71, "candidate_ms": 86, "candidate_error": False}, {"id": "deploy-09", "segment": "vip", "label": 1, "champion_prob": 0.80, "candidate_prob": 0.30, "champion_ms": 82, "candidate_ms": 210, "candidate_error": False}, {"id": "deploy-10", "segment": "vip", "label": 0, "champion_prob": 0.20, "candidate_prob": 0.70, "champion_ms": 79, "candidate_ms": 230, "candidate_error": True}, {"id": "deploy-11", "segment": "vip", "label": 1, "champion_prob": 0.70, "candidate_prob": 0.80, "champion_ms": 85, "candidate_ms": 190, "candidate_error": False}, {"id": "deploy-12", "segment": "vip", "label": 0, "champion_prob": 0.30, "candidate_prob": 0.20, "champion_ms": 81, "candidate_ms": 200, "candidate_error": False}]  # 定义十二条全局提升但 VIP 退化的模型请求。
champion_version = "fraud-v12"  # 定义当前稳定 champion 版本。
candidate_version = "fraud-v13"  # 定义待发布 candidate 版本。
print("教学实验输入：模型发布评测请求")  # 标记下方为离线可复现灰度数据。
print("请求       segment  label  champion/candidate  latency champion/candidate  candidate_error")  # 输出请求预览表头。
for request in requests:  # 逐条展示预测、延迟和错误语义。
    print(f"{request['id']:<10} {request['segment']:<8} {request['label']:>5} {request['champion_prob']:>8.2f}/{request['candidate_prob']:<8.2f} {request['champion_ms']:>8}/{request['candidate_ms']:<8} {request['candidate_error']}")  # 输出当前线上样本字段。

教学实验输入：模型发布评测请求
请求       segment  label  champion/candidate  latency champion/candidate  candidate_error
deploy-01  regular      1     0.80/0.90           72/84       False
deploy-02  regular      0     0.70/0.20           70/89       False
deploy-03  regular      1     0.40/0.80           75/93       False
deploy-04  regular      0     0.20/0.10           68/87       False
deploy-05  regular      1     0.70/0.80           80/96       False
deploy-06  regular      0     0.60/0.20           74/130      True
deploy-07  regular      1     0.80/0.90           76/91       False
deploy-08  regular      0     0.20/0.10           71/86       False
deploy-09  vip          1     0.80/0.30           82/210      False
deploy-10  vip          0     0.20/0.70           79/230      True
deploy-11  vip          1     0.70/0.80           85/190      False
deploy-12  vip          0     0.30/0.20           81/200      False


## 2. Baseline / 基线：只看全局离线准确率后全量发布

基线用 0.5 阈值计算全局准确率。Candidate 为 10/12，Champion 为 9/12，因此直接 100% 切流；它忽略了在线 timeout、p95 和 VIP 回归。

In [2]:
def predicted_label(probability):  # 用固定业务阈值把欺诈概率转为二分类。
    return int(probability >= 0.5)  # 返回零或一预测标签。
def offline_accuracy(events, probability_field, segment=None):  # 计算全局或指定分群离线准确率。
    selected = [event for event in events if segment is None or event["segment"] == segment]  # 选择目标流量分群。
    correct = sum(predicted_label(event[probability_field]) == event["label"] for event in selected)  # 统计预测正确数。
    return correct / len(selected)  # 返回样本准确率。
champion_accuracy = offline_accuracy(requests, "champion_prob")  # 计算 Champion 全局离线准确率。
candidate_accuracy = offline_accuracy(requests, "candidate_prob")  # 计算 Candidate 全局离线准确率。
baseline_promotes = candidate_accuracy > champion_accuracy  # 只要候选全局更高就直接全量发布。
baseline_rows = []  # 保存全量 Candidate 的逐请求线上结果。
for request in requests:  # 模拟十二条请求全部由 Candidate 服务。
    offline_correct = predicted_label(request["candidate_prob"]) == request["label"]  # 计算候选存档预测是否正确。
    online_success = offline_correct and not request["candidate_error"]  # 将真实在线错误纳入用户结果。
    baseline_rows.append({"id": request["id"], "served_model": candidate_version, "offline_correct": offline_correct, "online_success": online_success, "latency_ms": request["candidate_ms"], "error": request["candidate_error"]})  # 保存全量暴露结果。
baseline_error_exposure = sum(row["error"] for row in baseline_rows)  # 统计用户实际看到的候选错误。
print(f"Baseline全局准确率：champion={champion_accuracy:.1%}，candidate={candidate_accuracy:.1%}，promote={baseline_promotes}")  # 展示单指标发布理由。
print("Baseline全量发布结果")  # 标记下表展示真实错误和延迟。
print("请求       model       offline_correct  online_success  latency  error")  # 输出基线结果表头。
for row in baseline_rows:  # 逐请求展示全量 Candidate 暴露。
    print(f"{row['id']:<10} {row['served_model']:<11} {str(row['offline_correct']):>15} {str(row['online_success']):>15} {row['latency_ms']:>8} {str(row['error']):>6}")  # 输出当前请求的线上结果。

Baseline全局准确率：champion=75.0%，candidate=83.3%，promote=True
Baseline全量发布结果
请求       model       offline_correct  online_success  latency  error
deploy-01  fraud-v13              True            True       84  False
deploy-02  fraud-v13              True            True       89  False
deploy-03  fraud-v13              True            True       93  False
deploy-04  fraud-v13              True            True       87  False
deploy-05  fraud-v13              True            True       96  False
deploy-06  fraud-v13              True           False      130   True
deploy-07  fraud-v13              True            True       91  False
deploy-08  fraud-v13              True            True       86  False
deploy-09  fraud-v13             False           False      210  False
deploy-10  fraud-v13             False           False      230   True
deploy-11  fraud-v13              True            True      190  False
deploy-12  fraud-v13              True            True      200  False


## 3. 底层实现：Shadow 指标、稳定哈希 Canary 与发布状态机

Shadow 宽门禁先确认候选可运行；随后 40% 稳定 bucket 进入 Canary。推广门禁使用更严格的错误率、p95 和 VIP 回归，失败立即进入 ROLLBACK。

In [3]:
def percentile_higher(values, probability):  # 手写离散样本 higher percentile。
    ordered = sorted(values)  # 对延迟从小到大排序。
    index = max(0, min(len(ordered) - 1, math.ceil(probability * len(ordered)) - 1))  # 计算向上取整的零基索引。
    return ordered[index]  # 返回不插值的保守分位数。
def stable_bucket(request_id):  # 用 SHA-256 生成稳定的百分桶。
    digest_prefix = hashlib.sha256(request_id.encode("utf-8")).hexdigest()[:8]  # 取哈希前八位避免 Python hash 随进程变化。
    return int(digest_prefix, 16) % 100  # 映射到零到九十九的稳定 bucket。
shadow_metrics = {"offline_accuracy": candidate_accuracy, "error_rate": sum(request["candidate_error"] for request in requests) / len(requests), "p95_ms": percentile_higher([request["candidate_ms"] for request in requests], 0.95), "vip_accuracy": offline_accuracy(requests, "candidate_prob", "vip")}  # 汇总 Candidate 在全量 Shadow 流量上的指标。
shadow_pass = shadow_metrics["offline_accuracy"] >= champion_accuracy and shadow_metrics["error_rate"] <= 0.20 and shadow_metrics["p95_ms"] <= 250  # 使用宽门禁决定是否允许小流量 Canary。
canary_percent = 40  # 设定教学 Canary 流量比例。
canary_requests = [request for request in requests if stable_bucket(request["id"]) < canary_percent]  # 用稳定 bucket 选出用户可见 Candidate 请求。
canary_error_rate = sum(request["candidate_error"] for request in canary_requests) / len(canary_requests)  # 计算 Canary 真实在线错误率。
canary_p95_ms = percentile_higher([request["candidate_ms"] for request in canary_requests], 0.95)  # 计算 Canary p95 延迟。
champion_vip_accuracy = offline_accuracy(requests, "champion_prob", "vip")  # 计算 Champion VIP 准确率基线。
candidate_vip_accuracy = offline_accuracy(requests, "candidate_prob", "vip")  # 计算 Candidate VIP 准确率。
vip_regression = candidate_vip_accuracy - champion_vip_accuracy  # 计算关键分群相对回归。
promotion_checks = {"canary_error_rate<=10%": canary_error_rate <= 0.10, "canary_p95<=150ms": canary_p95_ms <= 150, "vip_regression>=-10%": vip_regression >= -0.10}  # 定义三个全量推广硬门禁。
promotion_pass = all(promotion_checks.values())  # 要求所有硬门禁同时通过。
deployment_ledger = [{"state": "SHADOW", "passed": shadow_pass, "metrics": shadow_metrics}, {"state": "CANARY", "passed": promotion_pass, "traffic_percent": canary_percent, "request_ids": [request["id"] for request in canary_requests], "checks": promotion_checks}, {"state": "ROLLBACK" if not promotion_pass else "PROMOTE", "active_version": champion_version if not promotion_pass else candidate_version, "reason": [name for name, passed in promotion_checks.items() if not passed]}]  # 保存完整发布状态迁移与原因。
print("Shadow与Canary状态迁移")  # 标记下表展示门禁中间量。
for event in deployment_ledger:  # 逐状态打印指标、流量和原因。
    print(event)  # 输出当前发布阶段证据。
print("Canary bucket：", [(request["id"], stable_bucket(request["id"]), request["segment"], request["candidate_error"]) for request in canary_requests])  # 展示稳定哈希选择和错误暴露。

Shadow与Canary状态迁移
{'state': 'SHADOW', 'passed': True, 'metrics': {'offline_accuracy': 0.8333333333333334, 'error_rate': 0.16666666666666666, 'p95_ms': 230, 'vip_accuracy': 0.5}}
{'state': 'CANARY', 'passed': False, 'traffic_percent': 40, 'request_ids': ['deploy-01', 'deploy-04', 'deploy-06', 'deploy-07', 'deploy-11', 'deploy-12'], 'checks': {'canary_error_rate<=10%': False, 'canary_p95<=150ms': False, 'vip_regression>=-10%': False}}
{'state': 'ROLLBACK', 'active_version': 'fraud-v12', 'reason': ['canary_error_rate<=10%', 'canary_p95<=150ms', 'vip_regression>=-10%']}
Canary bucket： [('deploy-01', 33, 'regular', False), ('deploy-04', 0, 'regular', False), ('deploy-06', 10, 'regular', True), ('deploy-07', 26, 'regular', False), ('deploy-11', 25, 'vip', False), ('deploy-12', 28, 'vip', False)]


## 4. 逐请求结果与结果解读

Canary 阶段只有稳定 bucket 内请求使用 Candidate，其余仍走 Champion；门禁失败后新请求全部切回 Champion。逐请求表展示实际暴露范围。

In [4]:
canary_ids = {request["id"] for request in canary_requests}  # 建立用户可见 Candidate 请求集合。
corrected_rows = []  # 保存 Canary 阶段十二条请求的实际模型与结果。
for request in requests:  # 逐请求应用稳定灰度路由。
    uses_candidate = request["id"] in canary_ids  # 检查当前用户是否进入 Canary bucket。
    probability = request["candidate_prob"] if uses_candidate else request["champion_prob"]  # 选择实际服务模型的预测。
    model = candidate_version if uses_candidate else champion_version  # 记录用户可见模型版本。
    error = request["candidate_error"] if uses_candidate else False  # 只有 Canary 用户可能暴露候选运行错误。
    online_success = predicted_label(probability) == request["label"] and not error  # 计算用户可见分类与运行结果。
    latency = request["candidate_ms"] if uses_candidate else request["champion_ms"]  # 读取实际路径延迟。
    corrected_rows.append({"id": request["id"], "bucket": stable_bucket(request["id"]), "served_model": model, "online_success": online_success, "latency_ms": latency, "error": error})  # 保存逐请求灰度结果。
corrected_error_exposure = sum(row["error"] for row in corrected_rows)  # 统计灰度阶段暴露的候选错误数。
rollback_triggered = deployment_ledger[-1]["state"] == "ROLLBACK"  # 检查最终状态是否自动回滚。
print("请求       bucket  served_model  online_success  latency  candidate_error")  # 输出逐请求灰度结果表头。
for row in corrected_rows:  # 逐条展示模型版本和用户结果。
    print(f"{row['id']:<10} {row['bucket']:>6}  {row['served_model']:<12} {str(row['online_success']):>14} {row['latency_ms']:>8} {str(row['error']):>16}")  # 输出当前请求真实暴露。
print(f"结果解读：全量发布会暴露{baseline_error_exposure}个Candidate运行错误；Canary仅暴露{corrected_error_exposure}个，随后rollback={rollback_triggered}，新流量恢复{champion_version}。")  # 解释灰度与回滚如何限制爆炸半径。

请求       bucket  served_model  online_success  latency  candidate_error
deploy-01      33  fraud-v13              True       84            False
deploy-02      56  fraud-v12             False       70            False
deploy-03      69  fraud-v12             False       75            False
deploy-04       0  fraud-v13              True       87            False
deploy-05      43  fraud-v12              True       80            False
deploy-06      10  fraud-v13             False      130             True
deploy-07      26  fraud-v13              True       91            False
deploy-08      82  fraud-v12              True       71            False
deploy-09      95  fraud-v12              True       82            False
deploy-10      52  fraud-v12              True       79            False
deploy-11      25  fraud-v13              True      190            False
deploy-12      28  fraud-v13              True      200            False
结果解读：全量发布会暴露2个Candidate运行错误；Canary仅暴露1个，随后rollback=T

## 5. 失败案例与修正：全局准确率掩盖 VIP 回归

Candidate 全局提升 8.3 个百分点，但 VIP 从 100% 降到 50%。只看全局会推广；分群门禁明确拒绝并触发回滚。

In [5]:
global_improvement = candidate_accuracy - champion_accuracy  # 计算候选全局离线提升。
aggregate_only_promotes = global_improvement > 0.0  # 复现只看全局准确率的推广决策。
segment_guard_promotes = aggregate_only_promotes and vip_regression >= -0.10  # 加入 VIP 最大允许回归门禁。
print(f"错误行为：global improvement={global_improvement:.1%}，aggregate_only_promotes={aggregate_only_promotes}，VIP回归被隐藏。")  # 展示辛普森式分群风险。
print(f"修正行为：champion_vip={champion_vip_accuracy:.1%}，candidate_vip={candidate_vip_accuracy:.1%}，regression={vip_regression:.1%}，promote={segment_guard_promotes}。")  # 展示关键分群门禁阻止推广。

错误行为：global improvement=8.3%，aggregate_only_promotes=True，VIP回归被隐藏。
修正行为：champion_vip=100.0%，candidate_vip=50.0%，regression=-50.0%，promote=False。


## 6. 生产边界

十二条样本不能做统计显著性判断。生产需要最小样本和置信区间、长期与短期 SLO、标签延迟对齐、用户/租户稳定哈希、版本化特征、配置中心原子切换、模型预热、缓存回滚兼容、告警抑制和定期回滚演练。

In [6]:
deployment_diagnostics = {"requests": len(requests), "champion_accuracy": champion_accuracy, "candidate_accuracy": candidate_accuracy, "candidate_vip_accuracy": candidate_vip_accuracy, "shadow_error_rate": shadow_metrics["error_rate"], "shadow_p95_ms": shadow_metrics["p95_ms"], "canary_requests": len(canary_requests), "baseline_error_exposure": baseline_error_exposure, "canary_error_exposure": corrected_error_exposure, "final_state": deployment_ledger[-1]["state"]}  # 汇总发布质量、风险和最终状态。
print("生产监控快照：", deployment_diagnostics)  # 输出模型发布平台应持续观察的指标。

生产监控快照： {'requests': 12, 'champion_accuracy': 0.75, 'candidate_accuracy': 0.8333333333333334, 'candidate_vip_accuracy': 0.5, 'shadow_error_rate': 0.16666666666666666, 'shadow_p95_ms': 230, 'canary_requests': 6, 'baseline_error_exposure': 2, 'canary_error_exposure': 1, 'final_state': 'ROLLBACK'}


## 7. 最小回归测试

断言覆盖样本规模、Shadow、稳定灰度、风险收敛、回滚和分群反例。

In [7]:
assert len(requests) >= 6 and {request["segment"] for request in requests} == {"regular", "vip"}  # 保证案例有足够请求和关键分群。
assert candidate_accuracy > champion_accuracy and baseline_promotes  # 保证单指标全量发布诱因真实存在。
assert shadow_pass and 0 < len(canary_requests) < len(requests)  # 保证 Shadow 通过且 Canary 确实只覆盖部分稳定流量。
assert all(stable_bucket(request["id"]) == next(row["bucket"] for row in corrected_rows if row["id"] == request["id"]) for request in requests)  # 保证相同 request_id 的灰度 bucket 可重复。
assert rollback_triggered and not promotion_pass and deployment_ledger[-1]["active_version"] == champion_version  # 保证硬指标失败触发自动回滚。
assert corrected_error_exposure <= baseline_error_exposure and len(canary_requests) < len(requests)  # 保证灰度限制候选错误暴露范围。
assert aggregate_only_promotes and not segment_guard_promotes and vip_regression == -0.5  # 保证全局提升掩盖 VIP 回归的失败分支真实执行。